# Phase 3 · Telugu (Model H) — Data Generation

Generates a **large** synthetic comparative / transitive reasoning dataset from a
**single unified name pool**, then performs a **random 80 / 10 / 10 split** into
train / val / test. Each example carries an `"answer"` field (the bare entity
name, without trailing punctuation) used for answer-only loss masking during
fine-tuning.

In [1]:
import json
import random
from pathlib import Path

OUTPUT_DIR = Path("../datasets/reasoning")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
print("Output directory:", OUTPUT_DIR.resolve())

Output directory: /Users/nikhilsai/Documents/Courses/LMA/SelfProject/individual-project-nikk8076/telugu/datasets/reasoning


## 1. Unified Entity Pool

A **single** pool of 65 Telugu names is used for all splits. After generation
the full dataset is shuffled and split randomly — every name may appear in any
partition. This matches real evaluation conditions and lets the model learn
*reasoning patterns* rather than memorising split-specific vocabularies.

In [2]:
# Unified Telugu name pool (65 names)
ALL_NAMES = [
    "రాజు",    "రమేష్",   "సీత",     "లక్ష్మి",  "వెంకట్",
    "అనిత",   "మోహన్",   "సావిత్రి", "కృష్ణ",   "సుధ",
    "రఘు",    "భానుమతి", "విజయ్",   "పద్మ",    "అర్జున్",
    "కమల",    "నవీన్",   "రత్న",    "ప్రకాష్",  "శాంతి",
    "హరి",    "దుర్గ",   "శ్రీనివాస్","మాల",    "ముకేష్",
    "వసంత",   "దినేష్",  "కళ",      "విక్రమ్",  "జ్యోతి",
    "సంతోష్",  "సరళ",    "బాలాజీ",  "రేఖ",    "ఆకాష్",
    "మీర",    "రోహిత్",  "నళిని",   "అభి",    "చంద్ర",
    "సురేష్",  "నిర్మల",  "గణేష్",   "ఇందిర",  "ప్రసాద్",
    "శైలజ",   "రవీందర్",  "ఉష",     "అజయ్",   "సుమిత్ర",
    "తేజ",    "మంజుల",   "అమర్",   "హేమ",    "దేవ్",
    "కాంత",   "విఘ్నేష్",  "స్వాతి",  "రాహుల్",  "ప్రభ",
    "అశోక్",  "విమల",   "చరణ్",   "కౌసల్య",  "దీప",
]
print(f"Unified name pool: {len(ALL_NAMES)} names")

Unified name pool: 65 names


## 2. Attribute Pools

In [3]:
ATTRIBUTES = {
    "height": {
        "noun": "ఎత్తు",
        "adj_more": "పొడవు", "adj_less": "పొట్టి",
        "unit": "సెంటీమీటర్లు",
        "question_more": "ఎవరు ఎక్కువ పొడవుగా ఉన్నారు",
        "question_less": "ఎవరు తక్కువ పొడవుగా ఉన్నారు",
    },
    "age": {
        "noun": "వయస్సు",
        "adj_more": "పెద్ద", "adj_less": "చిన్న",
        "unit": "సంవత్సరాలు",
        "question_more": "ఎవరు వయస్సులో పెద్దవారు",
        "question_less": "ఎవరు వయస్సులో చిన్నవారు",
    },
    "weight": {
        "noun": "బరువు",
        "adj_more": "భారీ", "adj_less": "తక్కువ బరువు",
        "unit": "కిలోగ్రాములు",
        "question_more": "ఎవరి బరువు ఎక్కువ",
        "question_less": "ఎవరి బరువు తక్కువ",
    },
    "price": {
        "noun": "ధర",
        "adj_more": "ఖరీదైనది", "adj_less": "చవకైనది",
        "unit": "రూపాయలు",
        "question_more": "ఏది ఎక్కువ ధరకు అమ్ముతారు",
        "question_less": "ఏది తక్కువ ధరకు అమ్ముతారు",
    },
    "speed": {
        "noun": "వేగం",
        "adj_more": "వేగంగా", "adj_less": "నెమ్మదిగా",
        "unit": "కి.మీ/గంట",
        "question_more": "ఎవరు ఎక్కువ వేగంగా నడిచారు",
        "question_less": "ఎవరు తక్కువ వేగంగా నడిచారు",
    },
}
print("Attributes:", list(ATTRIBUTES.keys()))

Attributes: ['height', 'age', 'weight', 'price', 'speed']


## 3. Template Functions

Each function returns `(prompt, completion, answer, metadata)` where:
- `completion` = full answer string with trailing period (e.g. `"కమల."`)
- `answer` = bare entity name for answer-only loss masking (e.g. `"కమల"`)

In [4]:
def t_direct_more(names, attr):
    a, b = names[0], names[1]
    va, vb = random.randint(10, 200), random.randint(10, 200)
    while va == vb: vb = random.randint(10, 200)
    prompt = (f"{a} యొక్క {attr['noun']} {va} {attr['unit']}. "
              f"{b} యొక్క {attr['noun']} {vb} {attr['unit']}. "
              f"{attr['question_more']}?")
    ans = a if va > vb else b
    return prompt, f"{ans}.", ans, {"type": "direct_more", "values": {a: va, b: vb}}

def t_direct_less(names, attr):
    a, b = names[0], names[1]
    va, vb = random.randint(10, 200), random.randint(10, 200)
    while va == vb: vb = random.randint(10, 200)
    prompt = (f"{a} యొక్క {attr['noun']} {va} {attr['unit']}. "
              f"{b} యొక్క {attr['noun']} {vb} {attr['unit']}. "
              f"{attr['question_less']}?")
    ans = b if va > vb else a
    return prompt, f"{ans}.", ans, {"type": "direct_less", "values": {a: va, b: vb}}

def t_transitive_largest(names, attr):
    a, b, c = names[0], names[1], names[2]
    prompt = (f"{a} యొక్క {attr['noun']} {b} కంటే ఎక్కువ. "
              f"{b} యొక్క {attr['noun']} {c} కంటే ఎక్కువ. "
              f"అందరిలో {attr['adj_more']} వ్యక్తి ఎవరు?")
    return prompt, f"{a}.", a, {"type": "transitive_largest", "order": [a, b, c]}

def t_transitive_smallest(names, attr):
    a, b, c = names[0], names[1], names[2]
    prompt = (f"{a} యొక్క {attr['noun']} {b} కంటే ఎక్కువ. "
              f"{b} యొక్క {attr['noun']} {c} కంటే ఎక్కువ. "
              f"అందరిలో {attr['adj_less']} వ్యక్తి ఎవరు?")
    return prompt, f"{c}.", c, {"type": "transitive_smallest", "order": [a, b, c]}

def t_transitive_relation(names, attr):
    a, b, c = names[0], names[1], names[2]
    prompt = (f"{a} యొక్క {attr['noun']} {b} కంటే ఎక్కువ. "
              f"{b} యొక్క {attr['noun']} {c} కంటే ఎక్కువ. "
              f"{a} మరియు {c} మధ్య {attr['noun']} విషయంలో ఎవరు ముందు?")
    return prompt, f"{a}.", a, {"type": "transitive_relation", "order": [a, b, c]}

def t_rank_three(names, attr):
    a, b, c = names[0], names[1], names[2]
    va, vb, vc = random.sample(range(10, 200), 3)
    ranked = sorted([(a, va), (b, vb), (c, vc)], key=lambda x: x[1], reverse=True)
    prompt = (f"{a} యొక్క {attr['noun']} {va} {attr['unit']}, "
              f"{b} యొక్క {attr['noun']} {vb} {attr['unit']}, "
              f"{c} యొక్క {attr['noun']} {vc} {attr['unit']}. "
              f"అందరిలో ఎక్కువ {attr['noun']} ఉన్న వ్యక్తి ఎవరు?")
    ans_name   = ranked[0][0]
    completion = f"{ans_name}."
    return prompt, completion, ans_name, {"type": "rank_three", "values": {a: va, b: vb, c: vc}}

def t_multihop_largest(names, attr):
    a, b, c, d = names[:4]
    prompt = (f"{a} యొక్క {attr['noun']} {b} కంటే ఎక్కువ. "
              f"{b} యొక్క {attr['noun']} {c} కంటే ఎక్కువ. "
              f"{c} యొక్క {attr['noun']} {d} కంటే ఎక్కువ. "
              f"అందరిలో అతిపెద్ద {attr['noun']} ఉన్న వ్యక్తి ఎవరు?")
    return prompt, f"{a}.", a, {"type": "multihop_largest", "order": [a, b, c, d]}

def t_multihop_smallest(names, attr):
    a, b, c, d = names[:4]
    prompt = (f"{a} యొక్క {attr['noun']} {b} కంటే ఎక్కువ. "
              f"{b} యొక్క {attr['noun']} {c} కంటే ఎక్కువ. "
              f"{c} యొక్క {attr['noun']} {d} కంటే ఎక్కువ. "
              f"అందరిలో అతి తక్కువ {attr['noun']} ఉన్న వ్యక్తి ఎవరు?")
    return prompt, f"{d}.", d, {"type": "multihop_smallest", "order": [a, b, c, d]}

TEMPLATES_2 = [t_direct_more, t_direct_less]
TEMPLATES_3 = [t_transitive_largest, t_transitive_smallest, t_transitive_relation, t_rank_three]
TEMPLATES_4 = [t_multihop_largest, t_multihop_smallest]
print(f"Templates: {len(TEMPLATES_2)} two-entity, {len(TEMPLATES_3)} three-entity, {len(TEMPLATES_4)} four-entity")

Templates: 2 two-entity, 4 three-entity, 2 four-entity


## 4. Dataset Generation + Random Split

Generate **20,000 examples** from the unified pool, shuffle with fixed seed,
then split **80 / 10 / 10** -> 16,000 train / 2,000 val / 2,000 test.

In [5]:
TOTAL   = 20_000
TRAIN_N = 16_000
VAL_N   = 2_000
TEST_N  = 2_000
SEED    = 42

random.seed(SEED)

def generate_all(name_pool, n):
    attr_keys = list(ATTRIBUTES.keys())
    examples  = []
    while len(examples) < n:
        attr_key = random.choice(attr_keys)
        attr     = ATTRIBUTES[attr_key]
        r        = random.random()
        if r < 0.30:
            func   = random.choice(TEMPLATES_2)
            sample = random.sample(name_pool, 2)
        elif r < 0.70:
            func   = random.choice(TEMPLATES_3)
            sample = random.sample(name_pool, 3)
        else:
            func   = random.choice(TEMPLATES_4)
            sample = random.sample(name_pool, 4)
        try:
            prompt, completion, answer, meta = func(sample, attr)
            examples.append({
                "prompt":      prompt,
                "completion":  completion,
                "answer":      answer,
                "language":    "te",
                "attribute":   attr_key,
                "template_id": func.__name__,
                "metadata":    meta,
            })
        except Exception:
            continue
    return examples

print(f"Generating {TOTAL:,} examples ...")
all_examples = generate_all(ALL_NAMES, TOTAL)
random.shuffle(all_examples)

train_ex = all_examples[:TRAIN_N]
val_ex   = all_examples[TRAIN_N : TRAIN_N + VAL_N]
test_ex  = all_examples[TRAIN_N + VAL_N :]

for split_name, exs in [("train", train_ex), ("val", val_ex), ("test", test_ex)]:
    out_path = OUTPUT_DIR / f"{split_name}.jsonl"
    with open(out_path, "w", encoding="utf-8") as f:
        for ex in exs:
            f.write(json.dumps(ex, ensure_ascii=False) + "\n")
    print(f"[{split_name:5s}] {len(exs):6,} examples  ->  {out_path}")

Generating 20,000 examples ...
[train] 16,000 examples  ->  ../datasets/reasoning/train.jsonl
[val  ]  2,000 examples  ->  ../datasets/reasoning/val.jsonl
[test ]  2,000 examples  ->  ../datasets/reasoning/test.jsonl


## 5. Dataset Statistics

In [6]:
print("=== Telugu Reasoning Dataset Statistics ===\n")
for split_name in ["train", "val", "test"]:
    path     = OUTPUT_DIR / f"{split_name}.jsonl"
    examples = [json.loads(l) for l in open(path, encoding="utf-8")]
    templates, attrs, seen_names = {}, {}, set()
    for ex in examples:
        templates[ex["template_id"]] = templates.get(ex["template_id"], 0) + 1
        attrs[ex["attribute"]]       = attrs.get(ex["attribute"], 0) + 1
        m = ex["metadata"]
        seen_names.update(m.get("order", []) or list(m.get("values", {}).keys()))
    print(f"{split_name} ({len(examples):,} examples, {len(seen_names)} unique names)")
    print(f"  Templates  : { {k: v for k, v in sorted(templates.items())} }")
    print(f"  Attributes : { {k: v for k, v in sorted(attrs.items())} }")
    print()

sample = json.loads(open(OUTPUT_DIR / "train.jsonl", encoding="utf-8").readline())
assert "answer" in sample, "Missing 'answer' field!"
print(f"'answer' field verified.")
print(f"  prompt    = '{sample['prompt'][:70]}...'")
print(f"  completion= '{sample['completion']}'")
print(f"  answer    = '{sample['answer']}'")

=== Telugu Reasoning Dataset Statistics ===

train (16,000 examples, 65 unique names)
  Templates  : {'t_direct_less': 2405, 't_direct_more': 2412, 't_multihop_largest': 2353, 't_multihop_smallest': 2434, 't_rank_three': 1636, 't_transitive_largest': 1635, 't_transitive_relation': 1524, 't_transitive_smallest': 1601}
  Attributes : {'age': 3251, 'height': 3135, 'price': 3260, 'speed': 3197, 'weight': 3157}

val (2,000 examples, 65 unique names)
  Templates  : {'t_direct_less': 312, 't_direct_more': 311, 't_multihop_largest': 298, 't_multihop_smallest': 293, 't_rank_three': 200, 't_transitive_largest': 194, 't_transitive_relation': 198, 't_transitive_smallest': 194}
  Attributes : {'age': 367, 'height': 403, 'price': 430, 'speed': 417, 'weight': 383}

test (2,000 examples, 65 unique names)
  Templates  : {'t_direct_less': 307, 't_direct_more': 309, 't_multihop_largest': 305, 't_multihop_smallest': 280, 't_rank_three': 205, 't_transitive_largest': 205, 't_transitive_relation': 197, 't_tr

## 6. Sample Examples

In [7]:
print("=== 6 Random Training Examples ===\n")
examples = [json.loads(l) for l in open(OUTPUT_DIR / "train.jsonl", encoding="utf-8")]
for ex in random.sample(examples, 6):
    print(f"[{ex['template_id']}] ({ex['attribute']})")
    print(f"  Prompt    : {ex['prompt']}")
    print(f"  Completion: {ex['completion']}   answer='{ex['answer']}'")
    print()

=== 6 Random Training Examples ===



[t_multihop_largest] (age)
  Prompt    : విజయ్ యొక్క వయస్సు నవీన్ కంటే ఎక్కువ. నవీన్ యొక్క వయస్సు దేవ్ కంటే ఎక్కువ. దేవ్ యొక్క వయస్సు సంతోష్ కంటే ఎక్కువ. అందరిలో అతిపెద్ద వయస్సు ఉన్న వ్యక్తి ఎవరు?
  Completion: విజయ్.   answer='విజయ్'

[t_direct_more] (height)
  Prompt    : లక్ష్మి యొక్క ఎత్తు 179 సెంటీమీటర్లు. దీప యొక్క ఎత్తు 75 సెంటీమీటర్లు. ఎవరు ఎక్కువ పొడవుగా ఉన్నారు?
  Completion: లక్ష్మి.   answer='లక్ష్మి'

[t_direct_less] (weight)
  Prompt    : ప్రకాష్ యొక్క బరువు 117 కిలోగ్రాములు. సుమిత్ర యొక్క బరువు 162 కిలోగ్రాములు. ఎవరి బరువు తక్కువ?
  Completion: ప్రకాష్.   answer='ప్రకాష్'

[t_rank_three] (height)
  Prompt    : రేఖ యొక్క ఎత్తు 137 సెంటీమీటర్లు, స్వాతి యొక్క ఎత్తు 125 సెంటీమీటర్లు, రోహిత్ యొక్క ఎత్తు 165 సెంటీమీటర్లు. అందరిలో ఎక్కువ ఎత్తు ఉన్న వ్యక్తి ఎవరు?
  Completion: రోహిత్.   answer='రోహిత్'

[t_rank_three] (age)
  Prompt    : రమేష్ యొక్క వయస్సు 152 సంవత్సరాలు, జ్యోతి యొక్క వయస్సు 143 సంవత్సరాలు, కాంత యొక్క వయస్సు 172 సంవత్సరాలు. అందరిలో ఎక్కువ వయస్సు ఉన్న వ్యక్తి ఎవర